# 06 · Report, checks, and release v1.0.0

**Goal:** regenerate every number from one code version, write the findings into the README, run the
checks GitHub will run, push, and publish the release.

In [ ]:
# Make the project importable; works in JupyterLab, VS Code and Colab without reinstalling.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src"))

%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from mapaudit import notebook as nb
from mapaudit import pipeline, report
from mapaudit.config import get_paths

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.3f}".format)
MODE = nb.notebook_mode()  # "real" for you; CI sets MAPAUDIT_NOTEBOOK_MODE=demo to test this notebook
print("Mode:", MODE)

## 1. Regenerate everything (final clean run, about 20 minutes)

In [ ]:
RERUN_ALL = False  # set to True for the final run
if MODE == "demo":
    nb.prepare_demo(through="simulate")
if RERUN_ALL:
    for step in (pipeline.build, pipeline.audit, pipeline.robustness, pipeline.simulate):
        step(MODE)
report.report(MODE)

## 2. Findings for the README

These sentences are generated from the tables. The cell writes them into `README.md` between the FINDINGS
markers and adds the headline figure. Then open README.md in JupyterLab (right-click → Open With → Editor)
and rewrite the bullets in your own words.

In [ ]:
print((get_paths(MODE).results / "headline.md").read_text())
if MODE == "real":
    nb.update_readme()
    print("README.md updated: now edit the wording.")

## 3. Checks before sharing

Formatting, lint, tests, website build, plus the pre-share check: no placeholders left and real results present.

In [ ]:
if MODE == "real":
    display(nb.run_checks(fix=True, release=True))

## Save your work to GitHub

With the checks above passing, commit and push what changed, with a message that says what you did.
Then watch the **Actions** tab on GitHub go green.

## 4. Publish the release on GitHub

1. Optional, first: connect the repository to Zenodo (zenodo.org → GitHub) to get a citable DOI for the
   release.
2. Set version 1.0.0 and today's date-released in CITATION.cff, commit and push, then publish release
   v1.0.0 (`gh release create v1.0.0`) with the v1.0.0 notes from the launch kit's `release_notes.md`.